# 計量つき TT ソルバの計測

`metric_tt.py` は CPU・float64 の参照実装。このノートは `metric_tt_solver.py` の device・精度・前処理を、Qwen3 の 1 層で比べる。

**予備実験は 2026-11-08 に終わった。`01_single_layer_pilot` のファイルは読むだけにする。** 読むファイルは `01_single_layer_pilot` の重み、較正 Gram、M3 コアだけである。書き込み先は `results/50_llm_tt_vs_matrix/10_qwen3_0p6b_base/02_metric_tt_solver_benchmark/` だけにする。

カーネルは `pytorch-svd`。

## 0. パスと環境

- 入力: プロジェクトルート。`data/50_llm_tt_vs_matrix/10_qwen3_0p6b_base/01_single_layer_pilot/weights_float64.pt`（キー `W`）。同ディレクトリの `calibration_grams.pt`（キー `total`、`n_tokens`）。`models/.../01_single_layer_pilot/m3_{T1,T2}_50.pt`（キー `cores`、`out_modes`、`in_modes`、`ranks`）。
- このステップで行うこと: 結果ディレクトリを作り、GPU 名と torch と CUDA の版を `environment.json` に残す。`G_λ` は `regularize_gram(total, 1e-6)`。mode は `select_balanced_tt_matrix_modes(3072, 1024, n_cores)` を計算し、保存済みの mode と一致することを確かめる。
- 行わないこと: モデルのロード、M5 の実行、`01_single_layer_pilot` への書き込み、TF32 をオンにした計測。
- 出力: 変数 `PROJECT`、`PILOT_DATA`、`PILOT_MODELS`、`RESULTS`、`WEIGHT`、`GRAM`、`CASES`。`environment.json`。
- 完了条件: `WEIGHT` が `(3072, 1024)` の float64 CPU。`GRAM` が `(1024, 1024)`。T1 は 4 コア、T2 は 3 コア。保存済み mode と再計算した mode が一致する。
- 次のステップへの接続: 1 は `CASES` と `GRAM` で、全サイトの HVP 時間を測る。

In [1]:
from pathlib import Path
import json
import statistics
import time

import pandas as pd
import torch

from nn_compression.compression.metric_low_rank import regularize_gram
from nn_compression.tt_mps.matrix_two_site import canonicalize_tt_matrix_cores
from nn_compression.tt_mps.metric_tt import (
    move_tt_matrix_center,
    run_metric_tt_als,
    select_balanced_tt_matrix_modes,
)
from nn_compression.tt_mps.metric_tt_solver import (
    MetricTTSolverConfig,
    metric_tt_hvp_on_device,
    run_metric_tt_als_on_device,
    solve_metric_tt_site_on_device,
)
from nn_compression.utils.artifacts import atomic_write_csv, atomic_write_json
from nn_compression.utils.paths import find_project_root, get_experiment_dirs

print("01_single_layer_pilot のファイルは読むだけにする。", flush=True)

PROJECT = find_project_root(Path.cwd())
PILOT_DATA = PROJECT / "data" / "50_llm_tt_vs_matrix" / "10_qwen3_0p6b_base" / "01_single_layer_pilot"
PILOT_MODELS = PROJECT / "models" / "50_llm_tt_vs_matrix" / "10_qwen3_0p6b_base" / "01_single_layer_pilot"
_data, _models, RESULTS = get_experiment_dirs(
    PROJECT,
    "50_llm_tt_vs_matrix",
    "10_qwen3_0p6b_base",
    "02_metric_tt_solver_benchmark",
    create=False,
)
RESULTS.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA が使えません。")

weights = torch.load(PILOT_DATA / "weights_float64.pt", map_location="cpu", weights_only=False)
grams = torch.load(PILOT_DATA / "calibration_grams.pt", map_location="cpu", weights_only=False)
WEIGHT = weights["W"]
GRAM = regularize_gram(grams["total"], 1e-6)
if WEIGHT.shape != (3072, 1024) or WEIGHT.dtype != torch.float64 or WEIGHT.device.type != "cpu":
    raise RuntimeError(f"重みの形式が違います: {tuple(WEIGHT.shape)} {WEIGHT.dtype} {WEIGHT.device}")
if GRAM.shape != (1024, 1024) or GRAM.dtype != torch.float64:
    raise RuntimeError(f"G_λ の形式が違います: {tuple(GRAM.shape)} {GRAM.dtype}")

CASES = {}
for label, n_cores in (("T1", 4), ("T2", 3)):
    payload = torch.load(PILOT_MODELS / f"m3_{label}_50.pt", map_location="cpu", weights_only=False)
    chosen = select_balanced_tt_matrix_modes(3072, 1024, n_cores)
    out_modes = tuple(int(mode) for mode in payload["out_modes"])
    in_modes = tuple(int(mode) for mode in payload["in_modes"])
    if out_modes != tuple(chosen["out_modes"]) or in_modes != tuple(chosen["in_modes"]):
        raise RuntimeError(f"{label} の保存済み mode が、偏り最小の再計算と違います。")
    cores = [core.detach().to(dtype=torch.float64, device="cpu").contiguous() for core in payload["cores"]]
    if len(cores) != n_cores:
        raise RuntimeError(f"{label} のコア数が {len(cores)} です。")
    CASES[label] = {"cores": cores, "out_modes": out_modes, "in_modes": in_modes, "budget": "50"}
    print(label, out_modes, in_modes, [tuple(core.shape) for core in cores], flush=True)

environment = {
    "gpu_name": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "matmul_tf32_before": bool(torch.backends.cuda.matmul.allow_tf32),
    "note": "計測中の TF32 は各セルで切る。M5 コアは使わない。短い ALS の初期値は M3 の 50% である。",
}
atomic_write_json(RESULTS / "environment.json", environment)
print(json.dumps(environment, ensure_ascii=False, indent=2), flush=True)

01_single_layer_pilot のファイルは読むだけにする。


T1 (2, 2, 24, 32) (16, 16, 2, 2) [(1, 2, 16, 32), (32, 2, 16, 382), (382, 24, 2, 64), (64, 32, 2, 1)]


T2 (2, 16, 96) (64, 8, 2) [(1, 2, 64, 109), (109, 16, 8, 109), (109, 96, 2, 1)]


{
  "gpu_name": "NVIDIA GeForce RTX 5070 Ti",
  "torch_version": "2.11.0+cu128",
  "cuda_version": "12.8",
  "matmul_tf32_before": false,
  "note": "計測中の TF32 は各セルで切る。M5 コアは使わない。短い ALS の初期値は M3 の 50% である。"
}


## 1. 1 回の HVP

- 入力: `CASES`、`GRAM`。T1 と T2 の 50% M3 コア。全サイト。
- このステップで行うこと: CPU float64、CUDA float64、CUDA float32 の 3 通りで、各サイトの HVP を 1 回温めてから 20 回測り、中央値を残す。float32 を含む 3 通りとも TF32 は切る。
- 行わないこと: ALS、前処理の構築、`G_λ` の低ランク近似。
- 出力: `hvp_times.csv`。列は `tensorization`、`budget`、`site`、`device`、`dtype`、`median_seconds`、`min_seconds`、`max_seconds`。変数 `HVP_FRAME`。
- 完了条件: T1 の 4 サイトと T2 の 3 サイトについて、3 通りずつ行がある。
- 次のステップへの接続: 2 は同じコアの中央サイトだけを使う。

In [2]:
def _sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def _median_hvp(cores, gram, site, out_modes, in_modes, repeats=20):
    direction = torch.randn(cores[site].shape, dtype=cores[site].dtype, device=cores[site].device)
    metric_tt_hvp_on_device(cores, site, direction, gram, out_modes, in_modes)
    _sync(cores[site].device)
    samples = []
    for _ in range(repeats):
        _sync(cores[site].device)
        started = time.perf_counter()
        metric_tt_hvp_on_device(cores, site, direction, gram, out_modes, in_modes)
        _sync(cores[site].device)
        samples.append(time.perf_counter() - started)
    return statistics.median(samples), min(samples), max(samples)


matmul_flag = torch.backends.cuda.matmul.allow_tf32
cudnn_flag = torch.backends.cudnn.allow_tf32
precision = torch.get_float32_matmul_precision()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
hvp_rows = []
try:
    placements = (
        ("cpu", torch.float64),
        ("cuda", torch.float64),
        ("cuda", torch.float32),
    )
    for label, case in CASES.items():
        for device_name, dtype in placements:
            moved_cores = [
                core.to(device=device_name, dtype=dtype) for core in case["cores"]
            ]
            moved_gram = GRAM.to(device=device_name, dtype=dtype)
            for site in range(len(moved_cores)):
                median_seconds, min_seconds, max_seconds = _median_hvp(
                    moved_cores,
                    moved_gram,
                    site,
                    case["out_modes"],
                    case["in_modes"],
                )
                hvp_rows.append(
                    {
                        "tensorization": label,
                        "budget": case["budget"],
                        "site": site,
                        "device": device_name,
                        "dtype": str(dtype).replace("torch.", ""),
                        "median_seconds": median_seconds,
                        "min_seconds": min_seconds,
                        "max_seconds": max_seconds,
                    }
                )
                print(
                    label,
                    site,
                    device_name,
                    str(dtype).replace("torch.", ""),
                    f"{median_seconds:.6f}",
                    flush=True,
                )
finally:
    torch.backends.cuda.matmul.allow_tf32 = matmul_flag
    torch.backends.cudnn.allow_tf32 = cudnn_flag
    torch.set_float32_matmul_precision(precision)

HVP_COLUMNS = [
    "tensorization",
    "budget",
    "site",
    "device",
    "dtype",
    "median_seconds",
    "min_seconds",
    "max_seconds",
]
atomic_write_csv(RESULTS / "hvp_times.csv", hvp_rows, HVP_COLUMNS)
HVP_FRAME = pd.DataFrame(hvp_rows, columns=HVP_COLUMNS)
if len(HVP_FRAME) != (4 + 3) * 3:
    raise RuntimeError(f"HVP の行数が {len(HVP_FRAME)} です。21 である必要があります。")
HVP_FRAME


T1 0 cpu float64 0.049168


T1 1 cpu float64 0.046256


T1 2 cpu float64 0.042870


T1 3 cpu float64 0.048678


T1 0 cuda float64 0.040182


T1 1 cuda float64 0.018162


... (12 outputs left out)

T2 0 cuda float32 0.000571


T2 1 cuda float32 0.000513


T2 2 cuda float32 0.000533


,tensorization,budget,site,device,dtype,median_seconds,min_seconds,max_seconds
0,T1,50,0,cpu,float64,0.049168,0.045800,0.057844
1,T1,50,1,cpu,float64,0.046256,0.037940,0.049007
2,T1,50,2,cpu,float64,0.042870,0.039735,0.047365
3,T1,50,3,cpu,float64,0.048678,0.044271,0.058808
4,T1,50,0,cuda,float64,0.040182,0.037647,0.040666
5,T1,50,1,cuda,float64,0.018162,0.017807,0.018794
6,T1,50,2,cuda,float64,0.018697,0.017537,0.019322
7,T1,50,3,cuda,float64,0.029053,0.028846,0.029516
8,T1,50,0,cuda,float32,0.000802,0.000785,0.001125
9,T1,50,1,cuda,float32,0.000665,0.000652,0.001011


## 2. 両端以外のサイトの局所問題

- 入力: `CASES` の 50% M3 コア、`WEIGHT`、`GRAM`。両端以外のすべてのサイト（T1 は site 1 と 2、T2 は site 1）。正準中心は site 0 に直してから、そのサイトへ移す。
- このステップで行うこと: Jacobi と Kronecker、float64 と float32_refine の 4 通りで、相対残差 `1e-2`、`1e-4`、`1e-6`、`1e-8` に初めて達した反復と時刻を残す。許容誤差は fixed の `1e-8`。CG は 500 回、float32 の外側反復は 5 回まで。設定は 1 つずつ実行する。局所問題の大きさ `K_k` の次元（`r_{k-1}·n_k·r_k`）と、前処理を作る時間・掛ける時間も残す。
- 行わないこと: 掃引、2 サイト更新、閾値に届かないときの許容誤差の変更。
- 出力: `local_convergence.csv`。列は `tensorization`、`budget`、`site`、`k_dimension`、`precision`、`preconditioner`、`threshold`、`history_index`、`seconds_to_reach`、`reached`、`final_relative_residual`、`iterations`、`refinements`、`seconds_total`、`seconds_preconditioner_setup`、`seconds_preconditioner_apply`、`kronecker_relative_error`、`kronecker_clipped`、`history_kind`。変数 `LOCAL_FRAME`。
- 完了条件: 3 サイト × 4 設定 × 4 閾値 = 48 行。届かない閾値は `reached` が False で、反復と時刻は空欄。
- 次のステップへの接続: 3 は同じ M3 コアから 5 掃引する。局所解のコアは使わない。

float64 の `history_index` は CG の反復で、0 は初期残差。float32_refine の履歴は外側反復の前後に測った float64 残差だけなので、`history_kind` は `outer_refinement`。内側 CG の float32 残差は入れない。

In [3]:
THRESHOLDS = (1e-2, 1e-4, 1e-6, 1e-8)
LOCAL_COLUMNS = [
    "tensorization",
    "budget",
    "site",
    "k_dimension",
    "precision",
    "preconditioner",
    "threshold",
    "history_index",
    "seconds_to_reach",
    "reached",
    "final_relative_residual",
    "iterations",
    "refinements",
    "seconds_total",
    "seconds_preconditioner_setup",
    "seconds_preconditioner_apply",
    "kronecker_relative_error",
    "kronecker_clipped",
    "history_kind",
]
local_rows = []
for label, case in CASES.items():
    canonical = canonicalize_tt_matrix_cores(case["cores"], case["out_modes"], case["in_modes"], 0)
    for site in range(1, len(canonical) - 1):
        cores = move_tt_matrix_center(canonical, case["out_modes"], case["in_modes"], 0, site)
        k_dimension = cores[site].shape[0] * case["in_modes"][site] * cores[site].shape[3]
        for precision in ("float64", "float32_refine"):
            for preconditioner in ("jacobi", "kronecker"):
                print(f"局所解 {label} site {site} K={k_dimension} {precision} {preconditioner}", flush=True)
                config = MetricTTSolverConfig(
                    device="cuda",
                    precision=precision,
                    preconditioner=preconditioner,
                    cg_tolerance_mode="fixed",
                    cg_relative_tolerance=1e-8,
                    max_cg_iterations=500,
                    max_refinements=5,
                )
                record = solve_metric_tt_site_on_device(
                    cores,
                    site,
                    WEIGHT,
                    GRAM,
                    case["out_modes"],
                    case["in_modes"],
                    config,
                )
                if len(record.residual_history) != len(record.seconds_history):
                    raise RuntimeError("残差と時刻の長さが違います。")
                history_kind = "cg_iteration" if precision == "float64" else "outer_refinement"
                for threshold in THRESHOLDS:
                    hit = next(
                        (index for index, residual in enumerate(record.residual_history) if residual <= threshold),
                        None,
                    )
                    local_rows.append(
                        {
                            "tensorization": label,
                            "budget": case["budget"],
                            "site": site,
                            "k_dimension": k_dimension,
                            "precision": precision,
                            "preconditioner": preconditioner,
                            "threshold": threshold,
                            "history_index": hit,
                            "seconds_to_reach": None if hit is None else record.seconds_history[hit],
                            "reached": hit is not None,
                            "final_relative_residual": record.final_relative_residual,
                            "iterations": record.iterations,
                            "refinements": record.refinements,
                            "seconds_total": record.seconds,
                            "seconds_preconditioner_setup": record.seconds_preconditioner_setup,
                            "seconds_preconditioner_apply": record.seconds_preconditioner_apply,
                            "kronecker_relative_error": record.kronecker_relative_error,
                            "kronecker_clipped": record.kronecker_clipped,
                            "history_kind": history_kind,
                        }
                    )
                print(
                    "  final",
                    f"{record.final_relative_residual:.3e}",
                    "iterations",
                    record.iterations,
                    "refinements",
                    record.refinements,
                    "seconds",
                    f"{record.seconds:.2f}",
                    "setup",
                    f"{record.seconds_preconditioner_setup:.3f}",
                    "apply",
                    f"{record.seconds_preconditioner_apply:.3f}",
                    "clipped",
                    record.kronecker_clipped,
                    flush=True,
                )

atomic_write_csv(RESULTS / "local_convergence.csv", local_rows, LOCAL_COLUMNS)
LOCAL_FRAME = pd.DataFrame(local_rows, columns=LOCAL_COLUMNS)
if len(LOCAL_FRAME) != 48:
    raise RuntimeError(f"局所問題の行数が {len(LOCAL_FRAME)} です。48 である必要があります。")
LOCAL_FRAME

局所解 T1 site 1 K=195584 float64 jacobi


  final 9.297e-09 iterations 323 refinements 0 seconds 6.38 setup 0.035 apply 0.012 clipped False


局所解 T1 site 1 K=195584 float64 kronecker


  final 6.789e-02 iterations 500 refinements 0 seconds 10.59 setup 0.168 apply 0.700 clipped False


局所解 T1 site 1 K=195584 float32_refine jacobi


  final 6.321e-09 iterations 340 refinements 2 seconds 0.53 setup 0.001 apply 0.009 clipped False


... (15 outputs left out)

  final 1.158e-12 iterations 1267 refinements 3 seconds 1.26 setup 0.001 apply 0.034 clipped False


局所解 T2 site 1 K=95048 float32_refine kronecker


  final 5.724e-02 iterations 2500 refinements 5 seconds 3.62 setup 0.091 apply 1.098 clipped False


,tensorization,budget,site,k_dimension,precision,preconditioner,threshold,history_index,seconds_to_reach,reached,final_relative_residual,iterations,refinements,seconds_total,seconds_preconditioner_setup,seconds_preconditioner_apply,kronecker_relative_error,kronecker_clipped,history_kind
0,T1,50,1,195584,float64,jacobi,1.000000e-02,34.0,0.767569,True,9.297489e-09,323,0,6.377410,0.035198,0.011716,NaN,False,cg_iteration
1,T1,50,1,195584,float64,jacobi,1.000000e-04,130.0,2.641128,True,9.297489e-09,323,0,6.377410,0.035198,0.011716,NaN,False,cg_iteration
2,T1,50,1,195584,float64,jacobi,1.000000e-06,226.0,4.471603,True,9.297489e-09,323,0,6.377410,0.035198,0.011716,NaN,False,cg_iteration
3,T1,50,1,195584,float64,jacobi,1.000000e-08,323.0,6.359219,True,9.297489e-09,323,0,6.377410,0.035198,0.011716,NaN,False,cg_iteration
4,T1,50,1,195584,float64,kronecker,1.000000e-02,NaN,NaN,False,6.788785e-02,500,0,10.591841,0.168125,0.699907,0.714990,False,cg_iteration
5,T1,50,1,195584,float64,kronecker,1.000000e-04,NaN,NaN,False,6.788785e-02,500,0,10.591841,0.168125,0.699907,0.714990,False,cg_iteration
6,T1,50,1,195584,float64,kronecker,1.000000e-06,NaN,NaN,False,6.788785e-02,500,0,10.591841,0.168125,0.699907,0.714990,False,cg_iteration
7,T1,50,1,195584,float64,kronecker,1.000000e-08,NaN,NaN,False,6.788785e-02,500,0,10.591841,0.168125,0.699907,0.714990,False,cg_iteration
8,T1,50,1,195584,float32_refine,jacobi,1.000000e-02,1.0,0.257141,True,6.320755e-09,340,2,0.525452,0.000906,0.008907,NaN,False,outer_refinement
9,T1,50,1,195584,float32_refine,jacobi,1.000000e-04,1.0,0.257141,True,6.320755e-09,340,2,0.525452,0.000906,0.008907,NaN,False,outer_refinement


## 3. 5 掃引の ALS（設定を 1 つずつ変える）

- 入力: `CASES` の M3 コア、`WEIGHT`、`GRAM`。T1 の 50% と T2 の 50%。
- このステップで行うこと: 次の 6 設定で、1 回ずつ 5 掃引する。掃引の相対減少では止めない（`sweep_tolerance=-1`）。adaptive の係数は既定値（`adaptive_eta=0.1`、上限 1e-2、下限 1e-8）。

| 名前 | 計算機 | 精度 | 前処理 | 許容誤差 |
|---|---|---|---|---|
| `cpu_reference` | CPU（`run_metric_tt_als`） | float64 | Jacobi | fixed 1e-8 |
| `cuda_default` | CUDA | float64 | Jacobi | fixed 1e-8 |
| `cuda_kronecker` | CUDA | float64 | Kronecker | fixed 1e-8 |
| `cuda_adaptive` | CUDA | float64 | Jacobi | adaptive |
| `cuda_float32` | CUDA | float32_refine | Jacobi | fixed 1e-8 |
| `cuda_all` | CUDA | float32_refine | Kronecker | adaptive |

- 行わないこと: 5 掃引を超える本実行、採用の決定、M5 コアの上書き、設定の同時実行。
- 出力: `als_trajectory.csv`。列は `tensorization`、`budget`、`solver`、`sweep`、`q_lambda`、`relative_decrease`、`seconds`、`n_rejected`、`cg_iterations`、`cg_iterations_total`、`max_local_gradient_norm`、`seconds_hvp`、`seconds_preconditioner`、`seconds_preconditioner_apply`、`seconds_qr`、`seconds_accept`、`peak_memory_bytes`、`stop_reason`。CPU 版は時間の内訳・メモリ・局所勾配が空欄。変数 `ALS_FRAME`。
- 完了条件: 2 条件 × 6 設定 × 5 掃引 = 60 行。最後の掃引だけ `stop_reason` が `max_sweeps`。
- 次のステップへの接続: 4 は保存済みの 3 の CSV を表にする。追加の計算はしない。

In [4]:
ALS_COLUMNS = [
    "tensorization",
    "budget",
    "solver",
    "sweep",
    "q_lambda",
    "relative_decrease",
    "seconds",
    "n_rejected",
    "cg_iterations",
    "cg_iterations_total",
    "max_local_gradient_norm",
    "seconds_hvp",
    "seconds_preconditioner",
    "seconds_preconditioner_apply",
    "seconds_qr",
    "seconds_accept",
    "peak_memory_bytes",
    "stop_reason",
]

DEVICE_JOBS = (
    ("cuda_default", dict(precision="float64", preconditioner="jacobi", cg_tolerance_mode="fixed")),
    ("cuda_kronecker", dict(precision="float64", preconditioner="kronecker", cg_tolerance_mode="fixed")),
    ("cuda_adaptive", dict(precision="float64", preconditioner="jacobi", cg_tolerance_mode="adaptive")),
    ("cuda_float32", dict(precision="float32_refine", preconditioner="jacobi", cg_tolerance_mode="fixed")),
    ("cuda_all", dict(precision="float32_refine", preconditioner="kronecker", cg_tolerance_mode="adaptive")),
)

als_rows = []
for label, case in CASES.items():
    print(f"cpu_reference {label}", flush=True)
    _cores, history = run_metric_tt_als(
        case["cores"],
        WEIGHT,
        GRAM,
        case["out_modes"],
        case["in_modes"],
        sweep_tolerance=-1.0,
        max_sweeps=5,
    )
    if len(history) != 5:
        raise RuntimeError(f"CPU {label} の掃引数が {len(history)} です。")
    for index, record in enumerate(history):
        print(f"  {label} cpu_reference sweep {record.sweep} Q {record.q_lambda:.6e} {record.seconds:.1f}s", flush=True)
        als_rows.append(
            {
                "tensorization": label,
                "budget": case["budget"],
                "solver": "cpu_reference",
                "sweep": record.sweep,
                "q_lambda": record.q_lambda,
                "relative_decrease": record.relative_decrease,
                "seconds": record.seconds,
                "n_rejected": record.n_rejected,
                "cg_iterations": ";".join(str(item) for item in record.cg_iterations),
                "cg_iterations_total": sum(record.cg_iterations),
                "max_local_gradient_norm": None,
                "seconds_hvp": None,
                "seconds_preconditioner": None,
                "seconds_preconditioner_apply": None,
                "seconds_qr": None,
                "seconds_accept": None,
                "peak_memory_bytes": None,
                "stop_reason": "max_sweeps" if index == len(history) - 1 else "",
            }
        )

    for solver_name, options in DEVICE_JOBS:
        print(f"{solver_name} {label}", flush=True)
        config = MetricTTSolverConfig(device="cuda", sweep_tolerance=-1.0, max_sweeps=5, **options)

        def _progress(record, solver_name=solver_name, label=label):
            print(
                f"  {label} {solver_name} sweep {record.sweep} Q {record.q_lambda:.6e} {record.seconds:.1f}s",
                flush=True,
            )

        _fitted, device_history = run_metric_tt_als_on_device(
            case["cores"],
            WEIGHT,
            GRAM,
            case["out_modes"],
            case["in_modes"],
            config,
            progress=_progress,
        )
        if len(device_history) != 5:
            raise RuntimeError(f"{solver_name} {label} の掃引数が {len(device_history)} です。")
        if device_history[-1].stop_reason != "max_sweeps":
            raise RuntimeError(f"{solver_name} {label} の停止理由が {device_history[-1].stop_reason} です。")
        if any(record.stop_reason is not None for record in device_history[:-1]):
            raise RuntimeError(f"{solver_name} {label} の途中の掃引に停止理由があります。")
        for record in device_history:
            als_rows.append(
                {
                    "tensorization": label,
                    "budget": case["budget"],
                    "solver": solver_name,
                    "sweep": record.sweep,
                    "q_lambda": record.q_lambda,
                    "relative_decrease": record.relative_decrease,
                    "seconds": record.seconds,
                    "n_rejected": record.n_rejected,
                    "cg_iterations": ";".join(str(item) for item in record.cg_iterations),
                    "cg_iterations_total": sum(record.cg_iterations),
                    "max_local_gradient_norm": max(record.local_gradient_norms),
                    "seconds_hvp": record.seconds_hvp,
                    "seconds_preconditioner": record.seconds_preconditioner,
                    "seconds_preconditioner_apply": record.seconds_preconditioner_apply,
                    "seconds_qr": record.seconds_qr,
                    "seconds_accept": record.seconds_accept,
                    "peak_memory_bytes": record.peak_memory_bytes,
                    "stop_reason": record.stop_reason or "",
                }
            )
        atomic_write_csv(RESULTS / "als_trajectory_partial.csv", als_rows, ALS_COLUMNS)

atomic_write_csv(RESULTS / "als_trajectory.csv", als_rows, ALS_COLUMNS)
(RESULTS / "als_trajectory_partial.csv").unlink(missing_ok=True)
ALS_FRAME = pd.DataFrame(als_rows, columns=ALS_COLUMNS)
if len(ALS_FRAME) != 60:
    raise RuntimeError(f"ALS の行数が {len(ALS_FRAME)} です。60 である必要があります。")
ALS_FRAME

cpu_reference T1


  T1 cpu_reference sweep 1 Q 2.457151e+07 50.9s


  T1 cpu_reference sweep 2 Q 2.328408e+07 28.0s


  T1 cpu_reference sweep 3 Q 2.273484e+07 26.6s


  T1 cpu_reference sweep 4 Q 2.242872e+07 25.7s


  T1 cpu_reference sweep 5 Q 2.223286e+07 24.9s


... (63 outputs left out)

  T2 cuda_all sweep 3 Q 3.262072e+07 8.7s


  T2 cuda_all sweep 4 Q 3.188738e+07 8.2s


  T2 cuda_all sweep 5 Q 3.139348e+07 8.2s


,tensorization,budget,solver,sweep,q_lambda,relative_decrease,seconds,n_rejected,cg_iterations,cg_iterations_total,max_local_gradient_norm,seconds_hvp,seconds_preconditioner,seconds_preconditioner_apply,seconds_qr,seconds_accept,peak_memory_bytes,stop_reason
0,T1,50,cpu_reference,1,2.457151e+07,0.737434,50.934692,0,13;405;240;0;0;362;0,1020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1,T1,50,cpu_reference,2,2.328408e+07,0.052395,27.990104,0,0;0;214;0;0;334;0,548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
2,T1,50,cpu_reference,3,2.273484e+07,0.023589,26.633472,0,0;0;208;0;0;312;0,520,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,T1,50,cpu_reference,4,2.242872e+07,0.013465,25.677466,0,0;0;203;0;0;296;0,499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
4,T1,50,cpu_reference,5,2.223286e+07,0.008732,24.900641,0,0;0;200;0;0;283;0,483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,max_sweeps
5,T1,50,cuda_default,1,2.457151e+07,0.737434,20.873797,0,13;405;240;0;0;361;0,1019,0.565355,19.536179,0.112502,0.039161,0.058875,0.203628,3.139507e+08,
6,T1,50,cuda_default,2,2.328408e+07,0.052395,11.563068,0,0;0;214;0;0;335;0,549,0.012430,10.460751,0.111886,0.024263,0.035024,0.201748,3.138406e+08,
7,T1,50,cuda_default,3,2.273484e+07,0.023589,11.035438,0,0;0;208;0;0;312;0,520,0.008500,9.925178,0.113053,0.022135,0.035747,0.206131,3.138406e+08,
8,T1,50,cuda_default,4,2.242872e+07,0.013465,10.564145,0,0;0;203;0;0;296;0,499,0.006426,9.502107,0.111724,0.020433,0.035842,0.202036,3.138406e+08,
9,T1,50,cuda_default,5,2.223286e+07,0.008732,10.312641,0,0;0;200;0;0;282;0,482,0.005162,9.222985,0.111796,0.020560,0.035396,0.204834,3.138406e+08,max_sweeps


## 4. 表

- 入力: `HVP_FRAME`、`LOCAL_FRAME`、`ALS_FRAME`。
- このステップで行うこと: HVP の中央値、閾値に達するまでの反復と時間を表にする。ALS は設定ごとに、5 掃引後の `Q_λ`、`cpu_reference` との `Q_λ` の相対差、合計時間、`cpu_reference` に対する速さの比を表にする。
- 行わないこと: 追加の HVP、局所解、ALS、ファイルの上書き。
- 出力: 画面上の 3 つの表。変数 `HVP_TABLE`、`LOCAL_TABLE`、`ALS_TABLE`。
- 完了条件: 3 つの表が、直前セルの CSV と同じ行数から集計されている。
- 次のステップへの接続: なし。採用の判断はしない。

In [5]:
from IPython.display import display

HVP_TABLE = HVP_FRAME.pivot_table(
    index=["tensorization", "site"],
    columns=["device", "dtype"],
    values="median_seconds",
)
LOCAL_TABLE = LOCAL_FRAME.loc[:, [
    "tensorization",
    "site",
    "k_dimension",
    "precision",
    "preconditioner",
    "threshold",
    "history_index",
    "seconds_to_reach",
    "reached",
    "final_relative_residual",
    "iterations",
    "seconds_total",
    "seconds_preconditioner_setup",
    "seconds_preconditioner_apply",
    "kronecker_clipped",
]]
final_sweep = ALS_FRAME.loc[ALS_FRAME["sweep"] == 5, [
    "tensorization",
    "solver",
    "q_lambda",
]].rename(columns={"q_lambda": "q_lambda_after_5"})
totals = ALS_FRAME.groupby(["tensorization", "solver"], as_index=False).agg(
    seconds_total=("seconds", "sum"),
    cg_iterations_total=("cg_iterations_total", "sum"),
)
ALS_TABLE = final_sweep.merge(totals, on=["tensorization", "solver"])
reference = ALS_TABLE.loc[ALS_TABLE["solver"] == "cpu_reference"].set_index("tensorization")
ALS_TABLE["q_relative_difference_vs_cpu"] = ALS_TABLE.apply(
    lambda row: (row["q_lambda_after_5"] - reference.loc[row["tensorization"], "q_lambda_after_5"])
    / reference.loc[row["tensorization"], "q_lambda_after_5"],
    axis=1,
)
ALS_TABLE["speedup_vs_cpu"] = ALS_TABLE.apply(
    lambda row: reference.loc[row["tensorization"], "seconds_total"] / row["seconds_total"],
    axis=1,
)
SOLVER_ORDER = ["cpu_reference", "cuda_default", "cuda_kronecker", "cuda_adaptive", "cuda_float32", "cuda_all"]
ALS_TABLE["solver"] = pd.Categorical(ALS_TABLE["solver"], SOLVER_ORDER, ordered=True)
ALS_TABLE = ALS_TABLE.sort_values(["tensorization", "solver"]).reset_index(drop=True)

print("HVP の中央値（秒）", flush=True)
display(HVP_TABLE)
print("中央サイトが閾値へ達するまで", flush=True)
display(LOCAL_TABLE)
print("5 掃引後の Q_λ と合計時間", flush=True)
display(ALS_TABLE)

HVP の中央値（秒）


device                   cpu      cuda          
dtype                float64   float32   float64
tensorization site                              
T1            0     0.049168  0.000802  0.040182
              1     0.046256  0.000665  0.018162
              2     0.042870  0.000630  0.018697
              3     0.048678  0.000729  0.029053
T2            0     0.034744  0.000571  0.014638
              1     0.035709  0.000513  0.014463
              2     0.032179  0.000533  0.016458

中央サイトが閾値へ達するまで


,tensorization,site,k_dimension,precision,preconditioner,threshold,history_index,seconds_to_reach,reached,final_relative_residual,iterations,seconds_total,seconds_preconditioner_setup,seconds_preconditioner_apply,kronecker_clipped
0,T1,1,195584,float64,jacobi,1.000000e-02,34.0,0.767569,True,9.297489e-09,323,6.377410,0.035198,0.011716,False
1,T1,1,195584,float64,jacobi,1.000000e-04,130.0,2.641128,True,9.297489e-09,323,6.377410,0.035198,0.011716,False
2,T1,1,195584,float64,jacobi,1.000000e-06,226.0,4.471603,True,9.297489e-09,323,6.377410,0.035198,0.011716,False
3,T1,1,195584,float64,jacobi,1.000000e-08,323.0,6.359219,True,9.297489e-09,323,6.377410,0.035198,0.011716,False
4,T1,1,195584,float64,kronecker,1.000000e-02,NaN,NaN,False,6.788785e-02,500,10.591841,0.168125,0.699907,False
5,T1,1,195584,float64,kronecker,1.000000e-04,NaN,NaN,False,6.788785e-02,500,10.591841,0.168125,0.699907,False
6,T1,1,195584,float64,kronecker,1.000000e-06,NaN,NaN,False,6.788785e-02,500,10.591841,0.168125,0.699907,False
7,T1,1,195584,float64,kronecker,1.000000e-08,NaN,NaN,False,6.788785e-02,500,10.591841,0.168125,0.699907,False
8,T1,1,195584,float32_refine,jacobi,1.000000e-02,1.0,0.257141,True,6.320755e-09,340,0.525452,0.000906,0.008907,False
9,T1,1,195584,float32_refine,jacobi,1.000000e-04,1.0,0.257141,True,6.320755e-09,340,0.525452,0.000906,0.008907,False


5 掃引後の Q_λ と合計時間


,tensorization,solver,q_lambda_after_5,seconds_total,cg_iterations_total,q_relative_difference_vs_cpu,speedup_vs_cpu
0,T1,cpu_reference,2.223286e+07,156.136374,3070,0.000000e+00,1.000000
1,T1,cuda_default,2.223286e+07,64.349089,3069,2.380997e-12,2.426396
2,T1,cuda_kronecker,2.231846e+07,252.010886,10471,3.850153e-03,0.619562
3,T1,cuda_adaptive,2.223359e+07,23.875094,943,3.309794e-05,6.539718
4,T1,cuda_float32,2.223286e+07,9.106827,4219,-7.311254e-11,17.144980
5,T1,cuda_all,2.224628e+07,83.577286,49030,6.039587e-04,1.868168
6,T2,cpu_reference,3.111547e+07,228.872495,5302,0.000000e+00,1.000000
7,T2,cuda_default,3.111547e+07,83.625549,5302,-2.676199e-10,2.736873
8,T2,cuda_kronecker,3.264725e+07,121.351934,6429,4.922912e-02,1.886023
9,T2,cuda_adaptive,3.112930e+07,11.379191,579,4.445650e-04,20.113248
